# Data processing

>Here we will process the data to start modeling.

In [1]:
import pandas as pd
import numpy as np
import regex as re
from nltk import download
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import pickle

## Data reading

>We read the data from full_data.csv from processed, wich is the merge of all the data treated with only the usefull information.

In [2]:
df = pd.read_csv('../data/processed/full_data.csv')
df.head()

,text,emotion
0,@tiffanylue i know i was listenin to bad habi...,sad
1,Layin n bed with a headache ughhhh...waitin o...,sad
2,Funeral ceremony...gloomy friday...,sad
3,wants to hang out with friends SOON!,joy
4,Re-pinging @ghostridah14: why didn't you go to...,fear


## Text processing functions
>Here we show all the functions we will use to process the text 

In [3]:

def preprocess_text(text):
    text = text.lower()
    #text = re.sub("https?|www", " ", text)
    text = re.sub(r'[^a-z ]', " ", text)
    text = re.sub(r'\s+', " ", text)
    return text.split()

In [4]:
download("wordnet")
lemmatizer = WordNetLemmatizer()

download("stopwords")
stop_words = stopwords.words("english")

def lemmatize_text(words, lemmatizer=lemmatizer):
    tokens = [lemmatizer.lemmatize(word) for word in words]
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [word for word in tokens if len(word) > 2]
    return tokens

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\adamc\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\adamc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Text processing
>Here we process the text with the functions above.

In [5]:
df["text"] = df["text"].apply(preprocess_text)
df.head()

,text,emotion
0,"[tiffanylue, i, know, i, was, listenin, to, ba...",sad
1,"[layin, n, bed, with, a, headache, ughhhh, wai...",sad
2,"[funeral, ceremony, gloomy, friday]",sad
3,"[wants, to, hang, out, with, friends, soon]",joy
4,"[re, pinging, ghostridah, why, didn, t, you, g...",fear


In [6]:
df["text"] = df["text"].apply(lemmatize_text)
df.head()

,text,emotion
0,"[tiffanylue, know, listenin, bad, habit, earli...",sad
1,"[layin, bed, headache, ughhhh, waitin, call]",sad
2,"[funeral, ceremony, gloomy, friday]",sad
3,"[want, hang, friend, soon]",joy
4,"[pinging, ghostridah, prom, like, friend]",fear


## Vectorize
>To finish with the text processing we will vectorize the texts into binary arrays that has a column for each concept and has a 1 if the text contains it and a 0 if it doesn't.

In [7]:
word_list = df["text"]
word_list = [" ".join(tokens) for tokens in word_list]

vectorizer = TfidfVectorizer(max_features=7500,
                             max_df=0.8,
                             min_df=5)

X = vectorizer.fit_transform(word_list).toarray()
y = df["emotion"]
X

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(496222, 7500))

>We also have to process the emotions because the comouter reads it in numbers and not in text, so we will use LabelEncoder

In [8]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
y

array([3, 3, 3, ..., 4, 4, 4], shape=(496222,))

>to see wich emotion goes with wich number we will use the label encoder to a dictionary with the emotions to show it

In [9]:
emotions = np.array(['joy', 'sad', 'anger', 'fear', 'surprise'])
emotions_label = label_encoder.transform(emotions)

emotions_dict = {}
for i in range(len(emotions)):
    emotions_dict[f'{emotions[i]}'] = emotions_label[i]

emotions_dict

{'joy': np.int64(2),
 'sad': np.int64(3),
 'anger': np.int64(0),
 'fear': np.int64(1),
 'surprise': np.int64(4)}

## Split
>To train and test the model we have to do an split of our data

In [10]:

X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=20)

In [11]:
X_train

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(396977, 7500))

In [12]:
X_test

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(99245, 7500))

In [13]:
y_train

array([2, 4, 1, ..., 1, 1, 2], shape=(396977,))

In [14]:
y_test

array([4, 2, 3, ..., 2, 3, 2], shape=(99245,))

## Saving

>First we are going to save the encoder and the vectorizer, the first one if you want to replicate the project and the second one to process new data.

In [15]:
with open('../models/emotions-encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)

with open('../models/text-vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)   

>Finally we will save the splitted data to train and test the model

In [16]:
with open('../data/processed/X-train.pkl', 'wb') as file:
    pickle.dump(X_train, file)

with open('../data/processed/X-test.pkl', 'wb') as file:
    pickle.dump(X_test, file)

with open('../data/processed/y-train.pkl', 'wb') as file:
    pickle.dump(y_train, file)

with open('../data/processed/y-test.pkl', 'wb') as file:
    pickle.dump(y_test, file)